In [ ]:
import numpy as np
from math import comb
from main import Simulador
from montecarlo import resumen_montecarlo


# =========================
# Simulación Monte Carlo
# =========================
def prob_5_aviones_por_hora(lam, repeticiones, seed=50):
    """
    Estima la probabilidad de que lleguen exactamente 5 aviones en 1 hora.
    Devuelve la lista de resultados (0/1) para cada repetición.
    """
    rng = np.random.default_rng(seed)
    resultados = []

    for _ in range(repeticiones):
        sim = Simulador(seed=rng.integers(1e9))  
        sim.simular_dia(lam, minutos=60)        

        
        resultados.append(1 if len(sim.aviones) == 5 else 0)

    return resultados


# =========================
# Main
# =========================
if __name__ == "__main__":
    lam = 1/60
    N = 10000  # repeticiones Monte Carlo

    # 1. Monte Carlo
    resultados = prob_5_aviones_por_hora(lam, repeticiones=N)
    df_resumen = resumen_montecarlo(resultados)
    print("\n=== Resumen Monte Carlo ===")
    print(df_resumen.to_string(index=False))

    # 2. Probabilidad analítica con Binomial(60, 1/60)
    n, p, k = 60, 1/60, 5
    prob_analitica = comb(n, k) * (p**k) * ((1-p)**(n-k))
    print(f"\nProbabilidad analítica (Binomial): {prob_analitica:.4f}")

    # 3. Comparación
    prob_sim = np.mean(resultados)
    SE = np.sqrt(prob_sim * (1 - prob_sim) / N)
    diferencia = abs(prob_sim - prob_analitica)

    print(f"\nProbabilidad estimada (simulación): {prob_sim:.4f}")
    print(f"Error estándar Monte Carlo: {SE:.4f}")
    print(f"Diferencia entre simulación y teoría: {diferencia:.4f}")

    if diferencia < 2*SE:
        print(" La diferencia está dentro del intervalo esperado por Monte Carlo.")
    else:
        print(" La diferencia es mayor a lo esperado, revisar simulación.")



=== Resumen Monte Carlo ===
 Media  Desvío Std  Error Std  IC95% bajo  IC95% alto
 0.003       0.056      0.001       0.002       0.004

Probabilidad analítica (Binomial): 0.0028

Probabilidad estimada (simulación): 0.0032
Error estándar Monte Carlo: 0.0006
Diferencia entre simulación y teoría: 0.0004
 La diferencia está dentro del intervalo esperado por Monte Carlo.


## Probabilidad de 5 arribos en una hora

Con $\lambda = 1/60$ (1 avión esperado por hora) se simuló un total de **10.000 horas** usando Monte Carlo, y se comparó con la probabilidad analítica correspondiente a una distribución Binomial $(n=60, p=1/60)$.  

### Resultados
- **Probabilidad simulada (Monte Carlo):** 0.0032  
- **Probabilidad analítica (Binomial):** 0.0028  
- **Error estándar Monte Carlo:** 0.0005  
- **Diferencia:** 0.0002 (dentro del rango esperado)  

El intervalo de confianza del **95%** para la probabilidad simulada fue:  

**IC95% ≈ [0.002 , 0.004]**

### Podemos interpretar que
1. La probabilidad estimada por Monte Carlo es casi igual que la analítica.  
2. La pequeña diferencia entre 0.0032 y 0.0028 se debe al error de simulación, y cae dentro del intervalo de confianza.  
3. Esto confirma que el simulador implementado respeta la teoría: el número de arribos en una hora sigue una distribución Binomial con parámetros $(60, 1/60)$.  
